In [ ]:
# ─── 1. Install ───────────────────────────────────────────────────────────────
!pip uninstall -y peft accelerate transformers -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  numpy==1.26.4 \
  Pillow

# ─── Restart runtime ──────────────────────────────────────────────────────────
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 16.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does no

In [ ]:
# ─── 2. Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/SpotFake_IFND"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── 3. Imports ───────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import numpy as np
import time
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from datasets import load_dataset, Image as HFImage

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 4. Load dataset ──────────────────────────────────────────────────────────
print("⏳ Loading dataset...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
hf_dataset = hf_dataset.cast_column("image", HFImage(decode=False))
print(hf_dataset)

# ─── 5. Transform & Dataset Class ─────────────────────────────────────────────
tokenizer  = BertTokenizer.from_pretrained("bert-base-uncased")
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class SpotFakeDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label  = int(sample['label'])
        text   = str(sample['text'])

        # ── Text ──────────────────────────────────────────────────────────
        inputs = tokenizer(
            text, truncation=True, padding="max_length",
            max_length=128, return_tensors="pt"
        )

        # ── Image ─────────────────────────────────────────────────────────
        try:
            img_bytes = sample['image']['bytes']
            if img_bytes:
                image = Image.open(BytesIO(img_bytes)).convert("RGB")
            else:
                image = Image.new("RGB", (224, 224), (128, 128, 128))
        except Exception:
            image = Image.new("RGB", (224, 224), (128, 128, 128))

        return {
            "input_ids":      inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
            "pixel_values":   img_transform(image),
            "labels":         torch.tensor(label, dtype=torch.long)
        }

train_dataset = SpotFakeDataset(hf_dataset['train'])
val_dataset   = SpotFakeDataset(hf_dataset['validation'])
test_dataset  = SpotFakeDataset(hf_dataset['test'])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 6. SpotFake+ Model ───────────────────────────────────────────────────────
class SpotFakePlus(nn.Module):
    def __init__(self):
        super().__init__()
        # Text branch — BERT
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.text_fc = nn.Sequential(
            nn.Linear(768, 32),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.4)
        )
        # Image branch — VGG19
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        vgg.classifier = nn.Identity()
        self.vgg = vgg
        self.img_fc = nn.Sequential(
            nn.Linear(25088, 32),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.4)
        )
        # Fusion classifier
        self.classifier = nn.Sequential(
            nn.Linear(64, 35),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.4),
            nn.Linear(35, 2)
        )

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        # Text
        text_out  = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = self.text_fc(text_out.pooler_output)
        # Image
        img_feat  = self.vgg(pixel_values)
        img_feat  = self.img_fc(img_feat)
        # Fusion
        fused     = torch.cat([text_feat, img_feat], dim=1)
        logits    = self.classifier(fused)
        loss      = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return loss, logits

model = SpotFakePlus().to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params/1e6:.2f}M")
print(f"Trainable params: {trainable_params/1e6:.2f}M")

# ─── 7. Training setup ────────────────────────────────────────────────────────
optimizer    = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps  = len(train_loader) * 5
warmup_steps = int(total_steps * 0.1)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps,
    num_training_steps=total_steps)

# ─── 8. Train & Eval functions ────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values   = batch["pixel_values"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        loss, _ = model(input_ids, attention_mask, pixel_values, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values   = batch["pixel_values"].to(device)
            labels         = batch["labels"]

            _, logits = model(input_ids, attention_mask, pixel_values)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            all_probs.extend(probs[:, 1])
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary")
    return {
        "accuracy":  accuracy_score(all_labels, all_preds),
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       roc_auc_score(all_labels, all_probs)
    }

# ─── 9. Training loop ─────────────────────────────────────────────────────────
best_f1   = 0
patience  = 0
best_path = os.path.join(SAVE_DIR, "best_model.pth")

print("\n🆕 Training from scratch")
for epoch in range(1, 6):
    train_loss  = train_epoch(model, train_loader, optimizer, scheduler)
    val_metrics = evaluate(model, val_loader)

    print(f"Epoch {epoch}/5 | Loss: {train_loss:.4f} | "
          f"F1: {val_metrics['f1']*100:.2f}% | "
          f"Acc: {val_metrics['accuracy']*100:.2f}% | "
          f"AUC: {val_metrics['auc']:.5f}")

    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        torch.save(model.state_dict(), best_path)
        print(f"  ✅ Best model saved (F1={best_f1*100:.2f}%)")
        patience = 0
    else:
        patience += 1
        if patience >= 2:
            print(f"  ⏹ Early stopping at epoch {epoch}")
            break

# ─── 10. Load best & Test ─────────────────────────────────────────────────────
model.load_state_dict(torch.load(best_path))
print("\n📊 Evaluating on test set...")
test_metrics = evaluate(model, test_loader)

print(f"\n{'='*40}")
print(f"Test Accuracy : {test_metrics['accuracy']*100:.2f}%")
print(f"Test Precision: {test_metrics['precision']*100:.2f}%")
print(f"Test Recall   : {test_metrics['recall']*100:.2f}%")
print(f"Test F1       : {test_metrics['f1']*100:.2f}%")
print(f"Test AUC      : {test_metrics['auc']:.5f}")
print(f"{'='*40}")

# ─── 11. Latency + VRAM ───────────────────────────────────────────────────────
model.eval()
dummy_ids    = torch.zeros(1, 128, dtype=torch.long).to(device)
dummy_mask   = torch.ones(1, 128, dtype=torch.long).to(device)
dummy_pixels = torch.randn(1, 3, 224, 224).to(device)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

with torch.no_grad():
    for _ in range(10):
        model(dummy_ids, dummy_mask, dummy_pixels)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(dummy_ids, dummy_mask, dummy_pixels)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0

print(f"\n{'='*40}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Latency median  : {np.median(latencies):.2f} ms")
print(f"VRAM (peak)     : {vram:.1f} MB")
print(f"{'='*40}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
⏳ Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

...

Train: 8416 | Val: 1052 | Test: 1053


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:03<00:00, 178MB/s]


Total params    : 130.34M
Trainable params: 130.34M

🆕 Training from scratch
Epoch 1/5 | Loss: 0.3508 | F1: 99.07% | Acc: 98.57% | AUC: 0.99733
  ✅ Best model saved (F1=99.07%)
Epoch 2/5 | Loss: 0.0809 | F1: 99.63% | Acc: 99.43% | AUC: 0.99796
  ✅ Best model saved (F1=99.63%)
Epoch 3/5 | Loss: 0.0382 | F1: 99.63% | Acc: 99.43% | AUC: 0.99646
Epoch 4/5 | Loss: 0.0248 | F1: 99.57% | Acc: 99.33% | AUC: 0.99726
  ⏹ Early stopping at epoch 4

📊 Evaluating on test set...

Test Accuracy : 99.05%
Test Precision: 99.26%
Test Recall   : 99.50%
Test F1       : 99.38%
Test AUC      : 0.99815

Total params    : 130,336,427
Trainable params: 130,336,427
Latency median  : 20.45 ms
VRAM (peak)     : 2056.1 MB
